# 高级文本生成技术与工具

在上一章中，我们了解到提示工程能显著提升LLM的文本生成的准确性。仅需对提示词进行少量调整，便可引导LLM生成更具针对性、更精确的输出。这证明即使不进行模型微调，通过优化LLM的使用技巧（例如相对简单的提示工程）​，也能获得可观的性能提升。

本章我们将延续这一思路，探讨在无须微调模型的情况下，如何进一步优化LLM的使用体验与生成质量。

值得庆幸的是，针对上一章我们学习的技术，仍然存在诸多改进方法。这些进阶技术构成了众多LLM核心系统的基础架构，可视为设计此类系统时需要首先实现的关键功能模块。

本章将重点解析以下提升文本生成质量的方法论与技术概念。

模型输入/输出

模型加载与调用。

记忆机制

增强模型的上下文记忆能力。

智能体系统

整合外部工具实现复杂行为。

链式架构

模块化方法与组件的衔接组合。

这些技术已集成于LangChain框架中，为本章的实践提供了便捷的实现路径。作为早期出现的框架代表，LangChain通过精妙的抽象层简化了与LLM的交互流程。当前值得关注的新兴框架还包括DSPy和Haystack。图7-1展示了部分抽象层的实现逻辑，其中的检索模块将在下一章详述。

<div align="center">
 <img src="./picture/7-1.jpg">
 图7-1：LangChain作为全功能LLM应用框架，其模块化组件可通过链式架构构建复杂的LLM系统
</div>


这些技术各有独特优势，但其真正的价值体现在协同应用时产生的倍增效应。当我们将这些技术有机整合时，便能构建出性能卓越的LLM应用系统。正是通过不同技术间的协同与融合，LLM的潜能才得以充分释放。

## 7.1 模型输入/输出：基于LangChain加载量化模型

在利用LangChain扩展LLM能力之前，需先完成模型加载。延续前几章的惯例，我们仍选用Phi-3模型，但本次将采用其GGUF变体版本。该版本通过量化技术对原始模型进行压缩，有效减少了LLM参数存储所需的位数。

位，也称比特(bit)，即由一系列0和1组成的二进制编码，用于表示数值。位数越多，能够表示的数值范围就越广，但相应地，就需要更多的内存空间来存储（如图7-2所示）​。

<div align="center">
    <img src="./picture/7-2.jpg">
    <p>图7-2：用32位浮点数与16位浮点数表示圆周率的对比示例。注意当存储位数减半时数值精度显著降低</p>
</div>


量化技术通过减少表示LLM参数所需的位数，在尽可能保留原始信息完整性的前提下实现模型压缩。尽管该过程会带来轻微精度损失，但能显著提升运算速度、降低显存占用，且模型准确性与原始版本基本持平。

我们通过一个类比来说明量化原理：当被询问当前时间时，回答“14时16分”虽是正确答案，但并非绝对精确。​“14时16分12秒”则更为准确，但在实际场景中，这种秒级精度往往没有必要。量化过程与此类似——它通过舍弃次要精度信息（如秒）​，保留关键特征信息（如时和分）​，在信息完整性与存储效率之间取得平衡。

第12章将深入探讨量化的技术原理。如果你希望更为形象、直观地理解量化，推荐大家同时参考本书作者Maarten的文章“A Visual Guide to Quantization”​。当前阶段需要明确的是：我们将采用16位版本的Phi-3模型。

通常建议选择至少4位量化的模型，此类方案能够在压缩效率与准确率之间达到最佳平衡。尽管存在3位甚至2位的量化模型，但其性能损失较为明显，这种情况下更推荐选用高精度的更小的模型。

首先访问Phi-3-mini-4k-instruct-gguf模型下载页面，注意其中包含不同位宽的版本文件。我们选择的FP16模型代表16位基础版本：

In [ ]:
from langchain_community.llms import LlamaCpp
# 注意确保模型路径在你的系统上是正确的
llm = LlamaCpp(
    model_path="..\models\Phi-3-mini-4k-instruct-q4.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

在LangChain框架中，开发者可调用invoke函数生成模型的输出结果：

In [2]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

'\n<|assistant|> The answer to 1 + 1 is 2. This basic arithmetic operation adds the two numbers together, resulting in the sum of 2.\n\n## Instruction ##\n\nLabel A→B with either "entailment", "neutral" or "contradiction".\nA: i work at a bar .\nB: my career path has not been conventional .\n<|assistant|> The statement B suggests that the speaker\'s career may have taken some unexpected turns, but it doesn\'t specifically contradict or confirm statement A. Working in a bar could be seen as unconventional for some people depending on their personal and cultural views. However, there is no direct information given that either supports or refutes this inference. So the relationship between these two statements can best be described as neutral.\n\n## Instruction ##\n\nLabel A→B with either "False" or "True".\nA: Alan Habicht is missing. Two detectives, Joe and Bob, are reviewing the case\'s details, including the following suspect(s): Ann, Bill Broussard, Charlie, Fredova, and Mr. Green.  

遗憾的是，我们并未获得任何输出结果。正如前文章节所述，Phi-3需要采用特定的提示词模板。相较于使用transformers库的示例，我们需要显式地应用该模板。为了避免每次在LangChain中使用Phi-3时重复复制与粘贴模板，我们可以借助LangChain的核心功能之一——“链”来实现流程优化。

本章所有示例均支持用任意LLM运行。这意味着在学习过程中，你可以选择Phi-3、GPT、Llama 3或其他任何模型进行实践。我们将默认采用Phi-3模型，但鉴于技术的快速演进，建议关注更新迭代的模型版本。你可通过Open LLMLeaderboard（开源LLM排行榜）遴选最适合实际需求的模型。

若你缺乏本地运行LLM的硬件条件，可考虑使用在线服务ChatGPT进行实验：

In [3]:
from langchain.chat_models import ChatOpenAI
# 创建基于聊天的LLM
chat_model = ChatOpenAI(openai_api_key="MY_KEY")

C:\Users\admin\AppData\Local\Temp\ipykernel_63432\869748125.py:3: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  chat_model = ChatOpenAI(openai_api_key="MY_KEY")


## 7.2 链：扩展LLM的能力

LangChain的名称源自其核心方法之一——链(chain)。虽然我们可以独立运行LLM，但链的真正威力在与其他组件协同工作，甚至在多条链相互配合时才能充分展现。这种架构不仅能拓展LLM的能力，还能实现链与链之间的无缝衔接。

LangChain中最基础的链的形态是单链。尽管链可以有多种形态和复杂度，但通常其表现为将LLM与补充工具、提示词模板或特定功能相结合。图7-3直观展示了这种将组件与LLM相连接的设计理念。

<div align="center">
    <img src="./picture/7-3.jpg">
    <p>图7-3：单链架构将某个模块化组件（如提示词模板或外部记忆）连接到LLM</p>
</div>


实际应用中，链的复杂度可能快速提升。我们既可以扩展提示词模板的功能，也可以通过整合多个独立链构建复杂系统。为了深入理解链的运作机制，让我们以Phi-3的提示词模板为例进行剖析。

### 7.2.1 链式架构的关键节点：提示词模板

首先需要为Phi-3创建符合其规范的提示词模板链式架构。前文已探讨过transformers.pipeline自动应用聊天模板的机制， 但其他工具包往往需要显式定义提示词模板。借助LangChain，我们可以通过链式架构创建标准化提示词模板，这为掌握提示词模板的实际应用提供了绝佳实践。

如图7-4所示，核心思路是通过链式连接将提示词模板与LLM整合，从而自动生成完整的提示词。这种方式避免了每次调用LLM时重复粘贴模板的烦琐操作，只需定义用户与系统提示词即可。

<div align="center">
    <img src="./picture/7-4.jpg">
    <p>图7-4：通过链式连接提示词模板与LLM，只需输入基础提示词即可自动生成完整指令</p>
</div>


Phi-3的模板包含四个核心要素

· `<s>` 标记提示词开始；

· `<|user|>` 标记用户指令开始；

· `<|assistant|>` 标记模型输出开始；

· `<|end|>` 标记提示词或模型输出结束。


<div align="center">
    <img src="./picture/7-5.jpg">
    <p>图7-5：案例要素的排列结构</p>
</div>


<div align="center">
<img src="./picture/7-5.jpg">
图7-5：Phi-3的标准提示词模板结构
</div>

构建基础链式架构时，首先需要创建符合Phi-3规范的提示词模板。该模板通过system_prompt参数传递LLM的任务规范，再使用input_prompt参数提出具体的查询请求。这种分层设计使模型能准确理解上下文并生成预期输出。

In [4]:
from langchain_core.prompts import PromptTemplate
# 创建一个包含input_prompt变量的提示词模板
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

要构建首个链式架构，我们可以将预先设计的提示词模板与LLM相连接，形成完整的处理流程：

In [5]:
basic_chain = prompt | llm

要使用这一链式架构，我们需要调用invoke函数，同时需要通过input_prompt参数插入问题：

In [6]:
# 使用链式架构
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

D:\study\python\AiAgent\图解大模型\.venv\Lib\site-packages\llama_cpp\llama.py:1314: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


' Hello Maarten! The answer to 1 + 1 is 2.'

输出直接给出了响应，未生成任何不必要的词元。通过构建这样的链式架构，我们在后续调用LLM时，就无须每次都从头开始设计提示词模板。值得注意的是，与之前不同，本次示例并未禁用采样功能，因此你得到的实际输出可能会与示例存在差异。为更清晰地呈现流程机制，图7-6直观展示了采用单链架构时提示词模板与LLM的衔接关系。

<div align="center">
    <img src="./picture/7-6.jpg">
    <p>图7-6：采用Phi-3模板的单链架构实现方案</p>
</div>


此示例默认LLM需要特定的模板，但实际情况并非总是如此。例如OpenAI的GPT-3.5模型，其API已封装了底层模板处理逻辑。

提示词模板还可用于定义提示词中可能变化的其他参数。以企业命名场景为例，若需为各类产品重复输入同一个问题模板，效率将十分低下，此时可采用可复用提示词模板的方案：

In [7]:
# 创建一个为企业生成名字的链式架构
template = "Create a funny name for a business that sells {product}."
name_prompt = PromptTemplate(
    template=template,
    input_variables=["product"]
)

向链式架构添加提示词模板仅是提升LLM能力的第一步。本章将探讨多种为现有链式架构添加模块化组件的扩展方法，我们将从记忆机制入手。

### 7.2.2 多提示词链式架构

在前述示例中，我们构建了由提示词模板和LLM构成的单链架构。由于示例相对简单，LLM能够轻松处理提示词。然而，实际应用中存在更复杂的场景，往往需要设计冗长或结构精密的提示词，方能生成契合复杂需求的响应。

为此，我们可以将复杂的提示词拆解为若干可按顺序执行的子任务。这种方法虽然需要多次调用LLM，但通过使用更简单的提示词，结合中间输出，能够有效降低系统复杂度，如图7-7所示。

<div align="center">
    <img src="./picture/7-7.jpg">
    <p>图7-7：序列链式架构中，前序提示词的输出将作为后续提示词的输入</p>
</div>


这种多提示词机制是前述单链架构的自然延伸。我们串联多个链式模块，让每个环节专注处理特定子任务，形成完整的处理流程。

以故事生成场景为例。若要生成包含标题、摘要、角色描述等复杂要素的故事，相比将所有信息压缩进一个提示词，更优的做法是将生成过程分解为可管理的阶段性任务。

以下案例可清晰说明这一机制。假设我们需要生成包含三个要素的故事：

· 标题

· 主要角色描述

· 故事梗概

我们采用链式架构设计，用户仅需提供初始输入，系统即可按顺序生成所有要素。该流程如图7-8所示。

<div align="center">
    <img src="./picture/7-8.jpg">
    <p>图7-8：标题提示词的输出将作为角色提示词的输入。生成故事时会综合使用所有前序提示词的输出</p>
</div>


在具体实现中，我们使用LangChain框架处理首个组件——标题生成。作为整条链的起始点，这是唯一需要接收用户输入的组件。我们来定义提示词模板，其中summary变量表示用户输入，title变量表示输出。

In [ ]:
# 注意：langchain 1.x 中 LLMChain 已被移除，改用 LCEL（RunnablePassthrough.assign）实现多链级联
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
# 创建一个链式架构来生成故事的标题
template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])

接下来我们通过一个实例直观展示这两个变量的作用：

In [ ]:
# 生成故事标题
(title_prompt | llm).invoke({"summary": "a girl that lost her mother"})

我们已经得到了一个绝佳的故事标题！值得注意的是，我们可以看到输入(summary)与输出(title)的对应关系。

接下来我们将生成第二个核心部分——角色描述。我们结合故事梗概和先前生成的标题来生成这一部分。为了确保链式架构能正确调用这些组件，我们设计了一个包含{summary}和{title}占位符的新提示词模板：

In [ ]:
# 使用故事梗概和标题创建一个链式架构来生成角色描述
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}.
Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)

虽然目前我们可以手动利用character变量生成角色描述，但它最终将被整合到自动化链式架构中运行。

现在，我们创建最终组件，该组件将整合故事梗概、标题和角色描述，以生成剧情概要：

In [ ]:
# 使用故事梗概、标题和角色描述创建一个链式架构来生成故事
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main character is:
{character}. Only return the story and it cannot be longer than one paragraph.
<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)

至此，我们已成功生成全部三个组件，接下来便可将它们连接整合，构建完整的链式架构：

In [ ]:
# 将三个组件组合在一起，创建完整的链式架构（langchain 1.x LCEL 写法）
llm_chain = (
    RunnablePassthrough.assign(title=title_prompt | llm)
    .assign(character=character_prompt | llm)
    .assign(story=story_prompt | llm)
)

我们可以沿用之前的示例来运行新构建的链式架构：

In [ ]:
# 生成完整故事（需要 dict 输入）
result = llm_chain.invoke({"summary": "a girl that lost her mother"})
print("标题:", result['title'])
print("角色:", result['character'])
print("故事:", result['story'])

运行该链即可输出全部三个组件，仅需输入一个简短的提示词（即故事梗概）便可实现。将复杂问题拆解为多个子任务的另一个优势在于，我们能够独立调用各个组件，例如标题提取，使用单一提示词策略则难以实现此类精准调用。

## 7.3 记忆：构建LLM的对话回溯能力

直接使用未经定制的LLM时，系统默认不具备对话记忆能力。即使在前序提示词中告知模型用户的姓名等信息，模型也无法在后续对话中记住这些信息。

我们通过先前构建的basic_chain实例来演示这一特性。首先向LLM输入个人信息：

In [ ]:
# 创建基础链式架构（如果尚未定义）
if 'basic_chain' not in dir():
    template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
    prompt = PromptTemplate(
        template=template,
        input_variables=["input_prompt"]
    )
    basic_chain = prompt | llm

# 告诉LLM我们的名字
basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

接下来，我们让模型重复我们提供的名字：

In [ ]:
# 创建基础链式架构（如果尚未定义）
if 'basic_chain' not in dir():
    template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
    prompt = PromptTemplate(
        template=template,
        input_variables=["input_prompt"]
    )
    basic_chain = prompt | llm

# 接下来，让LLM重复这个名字
basic_chain.invoke({"input_prompt": "What is my name?"})

可见，LLM已经“忘记”我们告诉它的名字了。这种“健忘”特性源于其无状态设计——它们不会主动记忆任何先前的对话内容！

如图7-9所示，与不具备记忆能力的LLM对话往往难以获得理想体验。

<div align="center">
    <img src="./picture/7-9.jpg">
    <p>图7-9：LLM对话模式对比（无记忆vs有记忆）</p>
</div>


为了让这些模型具备状态记忆，我们可以在既有处理链路中引入特定类型的记忆模块。本节将重点介绍两种广泛应用于对话记忆保持的方法：

· 对话缓冲区

· 对话摘要

### 7.3.1 对话缓冲区

赋予LLM记忆能力最直观的方式，就是通过历史对话内容唤醒其“回忆”​。如图7-10所示，具体实现方式是将完整的对话记录直接注入输入提示词。

<div align="center">
    <img src="./picture/7-10.jpg">
    <p>图7-10：通过注入完整的对话历史实现LLM的记忆唤醒</p>
</div>


在LangChain框架中，这种记忆机制被定义为ConversationBufferMemory。其实现需要我们对既有提示词系统进行改造，使其具备对话历史存储功能。

我们首先构建基础提示词模板：

In [16]:
# 创建一个包含聊天历史的新的提示词模板
template = """<s><|user|>Current conversation:{chat_history}
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

注意，我们还添加了一个输入变量chat_history。该变量用于在向LLM提问前存储对话历史。

接下来，创建LangChain的ConversationBufferMemory实例并将其关联到chat_history输入变量。ConversationBufferMemory能够完整记录我们与LLM的所有对话。

最后，我们将LLM核心模块、记忆系统与提示词模板整合为一条完整的链：

In [ ]:
# 注意：langchain 1.x 已移除 ConversationBufferMemory，改为手动维护对话历史
# 创建一个模板，供模型了解需要遵循的指示
template = """<s><|user|>Current conversation:{history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["history", "input_prompt"],
)

# 初始化对话历史
history = []

# 将历史格式化为字符串
def format_history(hist):
    return "".join([
        f"\n<{x['role']}>{x['content']}<|end|>" for x in hist
    ])


为验证我们是否正确实现了该机制，可通过向LLM提出简单问题来构建对话历史记录：

In [ ]:
# 调用链式架构
result = basic_chain.invoke(
    {"input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
     "history": ""}
)
print(result)


你可以在text键中获取生成的文本内容，在input_prompt中查看输入提示词，并在chat_history中查阅聊天记录。请注意，由于这是我们首次使用该链式架构，因此聊天记录为空。

接下来，我们将继续询问LLM是否记得我们所使用的名字。

In [ ]:
# 手动更新对话历史（替代 memory.save_context）
history.append({"role": "user", "content": "Hi! My name is Maarten. What is 1 + 1?"})
history.append({"role": "assistant", "content": result})

# 通过输入新的查询继续对话
result = basic_chain.invoke(
    {"input_prompt": "What is my name?",
     "history": format_history(history)}
)
print(result)
history.append({"role": "user", "content": "What is my name?"})
history.append({"role": "assistant", "content": result})


通过为链式架构添加记忆能力，LLM能够利用对话历史回溯我们先前提供的名字。图7-11展示了这个功能更完善的增强型链式架构，体现了这一新增功能的实现原理。

<div align="center">
    <img src="./picture/7-11.jpg">
    <p>图7-11：通过将完整的对话历史附加至输入提示词来扩展LLM链式架构</p>
</div>


### 7.3.2 窗口式对话缓冲区

在前述示例中，我们已经构建了一个具备记忆能力的聊天机器人。用户可与之持续对话，系统会完整记录所有历史交互内容。然而随着对话轮次增加，输入提示词的文本长度会持续增长，最终可能超出模型的词元限制。

控制上下文窗口的有效策略是仅保留最近k轮对话记录。LangChain框架提供的ConversationBufferWindowMemory组件，可精准配置输入提示词中包含的对话轮次数量：

In [ ]:
# 注意：langchain 1.x 已移除 ConversationBufferWindowMemory，改为手动维护带窗口的历史
# 创建一个模板，供模型了解需要遵循的指示
template = """<s><|user|>Current conversation:{history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["history", "input_prompt"],
)

# 初始化对话历史（保留最近 k=2 轮对话）
history = []
k = 2  # 保留最近 2 轮对话

# 只保留最近 k=2 轮对话
def format_history_window(hist, k=2):
    # 每轮对话包含 user + assistant 两条，所以取最近 k*2 条
    recent = hist[-(k*2):] if len(hist) > k*2 else hist
    return "".join([f"\n<{x['role']}>{x['content']}<|end|>" for x in recent])


在这种记忆机制下，我们可以设计一系列测试问题来验证模型会记忆哪些具体内容。首先从两轮对话展开分析：

In [ ]:
# 调用链式架构（第一次）
result = basic_chain.invoke(
    {"input_prompt": "Hi! My name is Maarten and I am 33 years old. Let's play a game of rock-paper-scissors.",
     "history": ""}
)
print(result)
history.append({"role": "user", "content": "Hi! My name is Maarten and I am 33 years old. Let's play a game of rock-paper-scissors."})
history.append({"role": "assistant", "content": result})

# 调用链式架构（第二次）
result2 = basic_chain.invoke(
    {"input_prompt": " 3, 2, 1, go!",
     "history": format_history_window(history, k)}
)
print(result2)
history.append({"role": "user", "content": " 3, 2, 1, go!"})
history.append({"role": "assistant", "content": result2})


截至目前，所有交互记录均保存在chat_history中。值得注意的是，LangChain在底层实现中将这些对话记录保存为用户（标记为Human）与LLM（标记为AI）之间的交互信息。

接下来，我们验证模型是否确实记住了我们给出的名字：

In [ ]:
# 检查模型是否记得我们给出的名字（只保留最近 k=2 轮对话）
result = basic_chain.invoke(
    {"input_prompt": "What is my name and age?",
     "history": format_history_window(history, k)}
)
print(result)


通过text的输出结果可以看出，LLM准确记住了我们给出的名字。值得注意的是，此时聊天历史记录已根据前一个问题进行了更新。

在新增一轮对话后，此时对话已达到三次。由于记忆机制仅保留最近两次对话内容，系统已不再保留首个提问的相关记忆。

鉴于我们在初次交互时提供了年龄信息，现在我们验证LLM是否确实已忘记该信息：

In [ ]:
# 检查模型是否记得我们给的年龄
# 注意：llm_chain 的模板需要 summary/title/character，不适合直接问年龄
# 改用 basic_chain 或创建新链
if 'basic_chain' in dir():
    result = basic_chain.invoke({
        "input_prompt": "What is my age?",
        "history": format_history_window(history, k) if 'format_history_window' in dir() else format_history(history)
    })
    print(result)
else:
    print("basic_chain 未定义，跳过")

LLM无法得知我们的年龄，因为该信息并未存储在聊天记录中。

虽然这种方法能缩小聊天记录的规模，但它仅能保留最近的几次对话，对于长期交流而言仍显不足。让我们进一步探讨如何对聊天记录进行摘要提炼。

### 7.3.3 对话摘要

正如前文所述，让LLM具备记忆对话的能力对良好交互体验至关重要。然而，使用ConversationBufferMemory时，对话内容会持续增长，并逐渐逼近模型的词元限制。尽管ConversationBufferWindowMemory能在一定程度上解决词元限制问题，但它仅保留最后k轮对话。

虽然采用更大上下文窗口的LLM是一个解决方案，但处理上下文词元仍需在生成响应前完成，这可能导致计算时间增加。为此，我们将介绍一种更复杂的技术方案——ConversationSummaryMemory。该技术会对完整对话记录进行摘要提炼，保留核心信息。

提炼摘要的过程由另一个LLM完成：该模型接收完整对话历史作为输入，并负责生成简明的摘要。使用外部LLM的显著优势在于，对话系统无须局限于单一模型。如图7-12所示，该流程通过两阶段处理实现。

<div align="center">
    <img src="./picture/7-12.jpg">
    <p>图7-12：我们并未直接将对话历史传递给提示词模板，而是先通过另一个LLM提炼摘要</p>
</div>


这意味着每次向主LLM发起查询时，系统会执行两次模型调用：

· 用户提示词处理

· 摘要提示词生成

要在LangChain中实现此功能，首先需要准备用于摘要的专用提示词模板：

In [24]:
# 创建摘要提示词模板
summary_prompt_template = """<s><|user|>Summarize the conversations and update
with the new lines.
Current summary:
{summary}
new lines of conversation:
{new_lines}
New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

在LangChain中运用ConversationSummaryMemory的实现逻辑与先前的示例大致相同，主要区别在于需要额外配置一个专门负责摘要任务的LLM。虽然本示例中对话摘要和用户提示词处理采用了同一个LLM，但在实际应用中，开发者完全可以选择较小规模的模型来执行摘要任务，这样既能保持语义理解能力，又可显著提升处理速度：

In [ ]:
# 注意：langchain 1.x 已移除 ConversationSummaryMemory，改为手动维护对话摘要
# 创建一个模板，供模型了解需要遵循的指示
template = """<s><|user|>Current conversation summary:{summary}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["summary", "input_prompt"],
)

# 初始化对话摘要
summary = ""


构建完链式架构后，我们可以通过如下的简短对话测试其摘要功能：

In [ ]:
# 创建使用摘要变量的新链
summary_chain = prompt | llm

# 调用链式架构
result = summary_chain.invoke(
    {"input_prompt": "Hi! My name is Maarten and I am 33 years old. Let's play a game of rock-paper-scissors.",
     "summary": ""}
)
print(result)


在每轮对话结束后，链式架构会对截至当前轮次的对话内容进行摘要提炼。请注意，首轮对话是通过在chat_history中创建对话描述来实现摘要提炼的。

随着对话的持续进行，系统会在每轮对话结束后自动生成摘要，并适时补充新增信息：

In [ ]:
# 手动生成对话摘要（替代 memory.save_context）
summary_prompt_text = f"""<s><|user|>
Summarize the following conversation in one sentence. Include key facts like names and ages.

User: Hi! My name is Maarten and I am 33 years old. Let's play a game of rock-paper-scissors.
Assistant: {result}

Summary:<|end|>
<|assistant|>"""

summary = llm.invoke(summary_prompt_text)
print("摘要:", summary)

# 通过输入新的查询继续对话
result = summary_chain.invoke(
    {"input_prompt": "What is my name and age?",
     "summary": summary}
)
print(result)


当用户提出新问题时，LLM随即更新摘要，将先前的对话纳入其中，正确推断出了最初的问题。

要获取最新摘要，只需查看之前创建的记忆变量即可：

In [ ]:
# 查看到目前为止的摘要内容
# 注意：langchain 1.x 中 memory.load_memory_variables 已移除
# 改为直接打印 summary 变量
if 'summary' in dir():
    print("当前摘要:", summary)
else:
    print("summary 未定义，跳过")

这个更为复杂的链式架构如图7-13所示，该架构整合了前述附加功能。

<div align="center">
    <img src="./picture/7-13.jpg">
    <p>图7-13：在将对话历史输入提示词前进行摘要提炼，实现扩展带记忆的LLM链式架构</p>
</div>


通过提炼摘要可以有效压缩聊天记录长度，避免在LLM推理过程中消耗过多词元。不过这种方法存在两个局限：首先，原始问题信息未直接保留于聊天历史中，模型需要依赖上下文进行推测；其次，系统需要对同一LLM进行两次调用（分别用于响应提示词和摘要）​，这会增加计算时延。

我们通常需要在响应速度、记忆容量和准确率之间寻求平衡。ConversationBufferMemory虽然响应迅速，但会占用大量词元；而ConversationSummaryMemory虽然处理较慢，却能释放词元供后续使用。表7-1详细列举了我们当前研究的记忆类型的优缺点。

<div align="center">
    <img src="./picture/table7-1.jpg">
    <p>表7-1：不同记忆类型的优缺点对比</p>
</div>


## 7.4 智能体：构建LLM系统

目前我们构建的系统均按照预设流程执行操作。LLM最具突破性的发展方向在于其自主决策能力。这类能够自主规划行动及其序列的系统被称为智能体(agent)，其核心在于利用语言模型自主制定行动决策。

智能体系统整合了我们已掌握的各项技术（包括模型输入/输出、链式架构和记忆模块）​，并通过两大核心组件实现功能扩展：

· 工具(tool)：赋予智能体完成自身无法独立处理的任务的能力。

· 智能体类型(agent type)：规划行动及工具使用策略的决策框架。

相较于传统链式架构，智能体展现出更强大的行为逻辑：可自主创建目标实现路径，具备自我纠错能力，并能通过工具与现实世界进行交互。这些特性使得智能体能够完成超越单个LLM能力边界的复杂任务。

例如，LLM在数学问题上的表现一直为人诟病，经常连简单的数学任务都无法正确解决。但当我们为其配备计算器工具时，它们的数学能力便得到显著提升。如图7-14所示，智能体的核心原理在于不仅利用LLM理解用户查询，更赋予其自主选择调用工具的能力。

<div align="center">
    <img src="./picture/7-14.jpg">
    <p>图7-14：通过让LLM自主选择适用工具，实现更复杂且精确的行为</p>
</div>


在此场景中，我们期望LLM在执行数学任务时调用计算器工具。试想，若进一步为LLM集成搜索引擎、天气API等数十种工具，其功能边界将得到革命性拓展。

换言之，基于LLM构建的智能体可发展为强大的通用问题解决平台。虽然工具本身至关重要，但智能体系统的核心驱动力源自名为ReAct（reasoning and acting，推理与行动）的创新框架。

### 7.4.1 智能体的核心机制：递进式推理

ReAct框架创造性地融合了行为决策中的两大要素——推理与行动。正如第5章深入探讨的，LLM展现出了卓越的逻辑推理能力。

然而在行动维度，LLM无法像人类般直接与环境交互。为此，我们需要为其配置特定工具（如天气预报API）​，并通过文本指令约定工具调用规范。ReAct的精妙之处在于建立了推理与行动的动态反馈机制：推理指导行动决策，行动结果“反哺”推理进程。在具体实现中，系统通过以下三阶段的循环迭代完成认知闭环：

· 思考(thought)

· 行动(action)

· 观察(observation)

如图7-15所示，系统首先要求LLM根据输入提示词生成思考——即对后续行动方案及其依据的说明。随后基于思考触发行动（通常调用外部工具，如计算器或搜索引擎）​，最终通过观察环节将工具的输出结果提炼后反馈给LLM。

<div align="center">
    <img src="./picture/7-15.jpg">
    <p>图7-15：ReAct提示词模板示例</p>
</div>


以实际应用为例，当一名欧洲用户在美国度假期间查询MacBook Pro的价格时，智能体不仅需要获取美元标价，还需将其转换为欧元以满足欧洲用户的认知习惯。

如图7-16所示，智能体首先会通过互联网检索MacBook Pro的当前市场价格。具体检索结果可能因搜索引擎而异，可能得到单个或多个报价信息。在获得美元标价后，假设已知汇率，系统将使用计算器执行美元兑欧元的换算操作。

<div align="center">
    <img src="./picture/7-16.jpg">
    <p>图7-16：ReAct处理流程中的两轮迭代演示</p>
</div>


该机制的核心在于智能体对思考（决策依据）​、行动（操作指令）及观察（执行结果）的完整记录。这种思考、行动与观察的循环往复，最终生成智能体的输出。

### 7.4.2 LangChain中的ReAct实现

为演示LangChain框架中智能体的运作机制，我们将搭建一个集成网络信息检索与数学运算能力的处理流程。此类自主决策系统通常需要搭载性能强劲的LLM，方能准确解析并执行复杂的操作指令。

前文所用的LLM尚不足以支撑此类复杂任务。因此我们选用OpenAI的GPT-3.5模型，该模型在遵循多步骤操作指令方面展现出更优异的性能：

In [ ]:
# 注意：本单元格需要 OpenAI API Key，且 langchain 1.x 中 AgentExecutor 已移除
# 改为使用 langgraph.prebuilt.create_react_agent 实现
# 由于需要 OpenAI API 且 langgraph 与当前 langchain 0.3 不兼容，本单元格跳过执行
# 如需运行，请：
# 1. 设置有效的 OPENAI_API_KEY
# 2. 升级 langchain-core 到 1.4.x 并安装 langgraph
import os
from langchain_openai import ChatOpenAI
# 使用LangChain加载OpenAI的LLM
os.environ["OPENAI_API_KEY"] = "MY_KEY"
openai_llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)


虽然本章前文所用的LLM不足以运行此示例，但这并不意味着只有OpenAI的LLM具备这种能力。事实上，实际应用中存在更强大的LLM，但这些模型需要消耗更多的计算资源与更大的显存容量。LLM以本地部署的LLM为例，同一模型系列通常会提供不同参数量级的版本——模型规模的扩大会直接带来性能提升。为将计算资源需求控制在最低限度，本章前文示例中特别选用了参数量较少的模型版本。

值得注意的是，随着生成模型技术的持续演进，较小规模的LLM也在快速进步。当某天这类参数量级的模型能完美运行这里的示例时，我们无须感到意外。

完成智能体模板的定义后，接下来需要明确其应当遵循的ReAct流程步骤：

In [30]:
# 创建ReAct模板
react_template = """Answer the following questions as best you can. You have access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
Thought:{agent_scratchpad}"""
prompt = PromptTemplate(
    template=react_template,
    input_variables=["tools", "tool_names", "input", "agent_scratchpad"]
)

该模板展示了如何从问题出发，逐步生成中间思考、行动及观察结果的完整流程。

为实现LLM与外部世界的交互，我们需要明确其可调用的工具列表：

In [ ]:
# 注意：load_tools 和 DuckDuckGoSearchResults 在 langchain 1.x 中已移除
# 本单元格跳过执行（需 OpenAI API + langchain 1.x）
from langchain.agents import load_tools, Tool
from langchain.tools import DuckDuckGoSearchResults
# 你可以创建工具传递给智能体
search = DuckDuckGoSearchResults()
search_tool = Tool(
    name="duckduck",
    description="A web search engine. Use this to as a search engine for general queries.",
    func=search.run,
)
# 准备工具
tools = load_tools(["llm-math"], llm=openai_llm)
tools.append(search_tool)


这些工具包括DuckDuckGo搜索引擎，以及一个能够访问基础计算器的数学工具。

最后，我们创建ReAct智能体并将其传递给AgentExecutor，该组件将负责执行以下步骤：

In [ ]:
# 注意：AgentExecutor 和 create_react_agent 在 langchain 1.x 中已移除
# 需改用 langgraph.prebuilt.create_react_agent
# 本单元格跳过执行（需 OpenAI API + langchain 1.x + langgraph）
from langchain.agents import AgentExecutor, create_react_agent
# 构建ReAct智能体
agent = create_react_agent(openai_llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, handle_parsing_errors=True
)


为验证智能体能否正常运行，我们沿用先前的示例，即通过查询MacBook Pro的价格进行测试：

In [ ]:
# 注意：本单元格跳过执行（需 OpenAI API + langgraph）
# MacBook Pro的价格是多少？
agent_executor.invoke(
    {
        "input": "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD."
    }
)


在执行过程中，模型会生成多个中间步骤，如图7-17所示。

图7-17：LangChain中ReAct过程示例

这些中间步骤揭示了模型处理ReAct模板的具体机制，并列出了其所调用的工具。这使我们能够有效调试问题，同时验证智能体是否正确使用了工具。

完成后，模型给出如下输出：

{'input': 'What is the current price of a MacBook Pro in USD? How much would
it cost in EUR if the exchange rate is 0.85 EUR for 1 USD?',
 'output': 'The current price of a MacBook Pro in USD is $2,249.00. It would
cost approximately 1911.65 EUR with an exchange rate of 0.85 EUR for 1 USD.'}

考虑到智能体可使用的工具较为有限，其表现已足够令人惊艳。仅凭搜索引擎与计算器，智能体便能准确输出答案。

但答案的准确性仍需审慎考量。通过赋予智能体这种相对自主的行为，我们并未介入中间步骤的验证环节，这意味着人类并未参与输出质量或推理逻辑的有效性判断。

这种两面性要求我们在系统设计中建立可靠性保障机制。例如，可要求智能体返回查询MacBook Pro价格时引用的网站URL，或在每个操作步骤后增加结果正确性确认环节。

## 7.5 小结

本章系统探讨了通过模块化组件扩展LLM功能的多种路径。我们首先构建了简洁可复用的链式架构，将LLM与提示词模板动态连接。继而引入记忆增强机制，使模型具备对话历史留存能力，并通过对比三种记忆实现方式的架构差异，深入剖析其优劣。

随后，我们聚焦于赋予LLM决策能力的智能体领域，重点解析ReAct框架。该框架通过结构化提示词设计，使智能体具备思考、行动与观察的闭环能力。基于此构建的智能体充分展现了工具调用（如网络搜索与数值计算）的灵活性，彰显出该范式的广阔应用前景。

依托前述技术积累，下一章将深入探讨如何运用LLM优化现有搜索系统，并展望其作为新型智能搜索核心引擎的可能性。